# M-MLP Baseline
External Baseline: Standard 4-Layer Vanilla MLP PINN


In [ ]:
# === Cell 1: Package Imports & Environment Setup ===
using Plots
using Flux
using NNlib
using LinearAlgebra
using Statistics
using Random
using CSV
using DataFrames
using Optim
using Printf
using JSON

println("Julia Environment Initialized. Packages Loaded Successfully!")


In [ ]:
# === Cell 2: Load Ground-Truth Hodgkin-Huxley Synthetic Dataset ===
file_path = raw"C:\nirbhay\Downloads\NeuroPinnsFormmer-attention-that-neurons-needs\Synthetic_Data\HH_ground_truth_synthetic_data.csv"
if !isfile(file_path)
    file_path = joinpath(@__DIR__, "..", "Synthetic_Data", "HH_ground_truth_synthetic_data.csv")
end

HH_data = CSV.read(file_path, DataFrame)
println("Loaded Hodgkin-Huxley dataset. Total timesteps: ", size(HH_data, 1))
first(HH_data, 5)


In [ ]:
# === Cell 3: Biophysical Constants & HH Rate Equations ===
const C_M   = 1.0f0      # Membrane Capacitance (uF/cm^2)
const G_NA  = 120.0f0    # Max Na+ Conductance (mS/cm^2)
const G_K   = 36.0f0     # Max K+ Conductance (mS/cm^2)
const G_L   = 0.3f0      # Leak Conductance (mS/cm^2)
const E_NA  = 50.0f0     # Na+ Reversal Potential (mV)
const E_K   = -77.0f0    # K+ Reversal Potential (mV)
const E_L   = -54.4f0    # Leak Reversal Potential (mV)

const IC_GROUND_TRUTH = Float32[-65.0, 0.0529, 0.5961, 0.3177]

function alpha_m(V::Real)
    x = -(V + 40.0f0) / 10.0f0
    return iszero(x) ? 1.0f0 : (0.1f0 * (V + 40.0f0)) / (-expm1(x))
end

function beta_m(V::Real)
    return 4.0f0 * exp(-(V + 65.0f0) / 18.0f0)
end

function alpha_h(V::Real)
    return 0.07f0 * exp(-(V + 65.0f0) / 20.0f0)
end

function beta_h(V::Real)
    return 1.0f0 / (1.0f0 + exp(-(V + 35.0f0) / 10.0f0))
end

function alpha_n(V::Real)
    x = -(V + 55.0f0) / 10.0f0
    return iszero(x) ? 0.1f0 : (0.01f0 * (V + 55.0f0)) / (-expm1(x))
end

function beta_n(V::Real)
    return 0.125f0 * exp(-(V + 65.0f0) / 80.0f0)
end


In [ ]:
# === Cell 4: Standalone Vanilla MLP PINN Architecture Definition ===
struct VanillaMLP
    net::Flux.Chain
end
Flux.@layer VanillaMLP

function VanillaMLP(; hidden_dim::Int=64, num_layers::Int=4, act_fn=tanh)
    layers = []
    push!(layers, Flux.Dense(1 => hidden_dim, act_fn))
    for _ in 1:(num_layers - 2)
        push!(layers, Flux.Dense(hidden_dim => hidden_dim, act_fn))
    end
    push!(layers, Flux.Dense(hidden_dim => 4))
    return VanillaMLP(Flux.Chain(layers...))
end

function (m::VanillaMLP)(t_batch)
    in_dims = ndims(t_batch)
    if in_dims == 3
        B = size(t_batch, 3)
        t_flat = reshape(t_batch, 1, B)
        raw_out = m.net(t_flat)
        V = -20.0f0 .+ 70.0f0 .* tanh.(raw_out[1:1, :])
        gating = NNlib.sigmoid.(raw_out[2:4, :])
        out_flat = cat(V, gating; dims=1)
        return reshape(out_flat, 4, 1, B)
    else
        raw_out = m.net(t_batch)
        V = -20.0f0 .+ 70.0f0 .* tanh.(raw_out[1:1, :])
        gating = NNlib.sigmoid.(raw_out[2:4, :])
        return cat(V, gating; dims=1)
    end
end


In [ ]:
# === Cell 5: AD Derivative Extraction & Vectorized Loss Engine ===
function compute_ad_derivatives(model, t_tensor; δt=1f-4)
    X_pred = model(t_tensor)
    X_plus  = model(t_tensor .+ δt)
    X_minus = model(t_tensor .- δt)
    dX_dt = (X_plus .- X_minus) ./ (2.0f0 * δt)
    return X_pred, dX_dt
end

function compute_physics_residuals(X_pred, dX_pred_dt, I_ext_value=10.0f0)
    V = X_pred[1, :, :]
    m = X_pred[2, :, :]
    h = X_pred[3, :, :]
    n = X_pred[4, :, :]

    I_Na = G_NA .* (m .^ 3) .* h .* (V .- E_NA)
    I_K  = G_K  .* (n .^ 4) .* (V .- E_K)
    I_L  = G_L  .* (V .- E_L)

    f_V = (I_ext_value .- I_Na .- I_K .- I_L) ./ C_M
    f_m = alpha_m.(V) .* (1.0f0 .- m) .- beta_m.(V) .* m
    f_h = alpha_h.(V) .* (1.0f0 .- h) .- beta_h.(V) .* h
    f_n = alpha_n.(V) .* (1.0f0 .- n) .- beta_n.(V) .* n

    R_V = dX_pred_dt[1, :, :] .- f_V
    R_m = dX_pred_dt[2, :, :] .- f_m
    R_h = dX_pred_dt[3, :, :] .- f_h
    R_n = dX_pred_dt[4, :, :] .- f_n

    L_RV = mean(abs2, R_V)
    L_Rm = mean(abs2, R_m)
    L_Rh = mean(abs2, R_h)
    L_Rn = mean(abs2, R_n)

    return L_RV, L_Rm, L_Rh, L_Rn
end

function compute_boundary_and_data_losses(model, X_pred, ground_truth_batch)
    t_zero = zeros(Float32, 1, 1, 1)
    X_zero = model(t_zero)[:, 1, 1]
    L_ic = mean(abs2, X_zero .- IC_GROUND_TRUTH)
    L_data = mean(abs2, X_pred .- ground_truth_batch)
    return L_ic, L_data
end

mutable struct NTKState
    RV::Float32; Rm::Float32; Rh::Float32; Rn::Float32; ic::Float32; data::Float32
end
NTKState() = NTKState(1.0f0, 1.0f0, 1.0f0, 1.0f0, 1.0f0, 1.0f0)

function update_ntk_weights_ema!(λ_state, model, batch_t, batch_data, I_ext; α=0.1f0)
    _, gs_RV   = Flux.withgradient(m -> compute_physics_residuals(compute_ad_derivatives(m, batch_t)[1], compute_ad_derivatives(m, batch_t)[2], I_ext)[1], model)
    _, gs_Rm   = Flux.withgradient(m -> compute_physics_residuals(compute_ad_derivatives(m, batch_t)[1], compute_ad_derivatives(m, batch_t)[2], I_ext)[2], model)
    _, gs_Rh   = Flux.withgradient(m -> compute_physics_residuals(compute_ad_derivatives(m, batch_t)[1], compute_ad_derivatives(m, batch_t)[2], I_ext)[3], model)
    _, gs_Rn   = Flux.withgradient(m -> compute_physics_residuals(compute_ad_derivatives(m, batch_t)[1], compute_ad_derivatives(m, batch_t)[2], I_ext)[4], model)
    _, gs_ic   = Flux.withgradient(m -> compute_boundary_and_data_losses(m, m(batch_t), batch_data)[1], model)
    _, gs_data = Flux.withgradient(m -> compute_boundary_and_data_losses(m, m(batch_t), batch_data)[2], model)

    function trace_norm(gs)
        if isnothing(gs) || isnothing(gs[1]); return 1.0f0; end
        flat_g, _ = Flux.destructure(gs[1])
        return sum(abs2, flat_g) + 1f-8
    end

    tr_RV, tr_Rm, tr_Rh = trace_norm(gs_RV), trace_norm(gs_Rm), trace_norm(gs_Rh)
    tr_Rn, tr_ic, tr_data = trace_norm(gs_Rn), trace_norm(gs_ic), trace_norm(gs_data)

    mean_target = (tr_RV + tr_Rm + tr_Rh + tr_Rn + tr_ic + tr_data) / 6.0f0
    sum_λ = (mean_target/tr_RV) + (mean_target/tr_Rm) + (mean_target/tr_Rh) + (mean_target/tr_Rn) + (mean_target/tr_ic) + (mean_target/tr_data)
    
    λ_state.RV   = (1.0f0 - α) * λ_state.RV   + α * (6.0f0 * (mean_target/tr_RV) / sum_λ)
    λ_state.Rm   = (1.0f0 - α) * λ_state.Rm   + α * (6.0f0 * (mean_target/tr_Rm) / sum_λ)
    λ_state.Rh   = (1.0f0 - α) * λ_state.Rh   + α * (6.0f0 * (mean_target/tr_Rh) / sum_λ)
    λ_state.Rn   = (1.0f0 - α) * λ_state.Rn   + α * (6.0f0 * (mean_target/tr_Rn) / sum_λ)
    λ_state.ic   = (1.0f0 - α) * λ_state.ic   + α * (6.0f0 * (mean_target/tr_ic) / sum_λ)
    λ_state.data = (1.0f0 - α) * λ_state.data + α * (6.0f0 * (mean_target/tr_data) / sum_λ)
    return λ_state
end

function total_loss_causal(model, batch_t, batch_data, λ, I_ext; ϵ=1.0f0, use_causality=true)
    X_pred, dX_dt = compute_ad_derivatives(model, batch_t)
    V, m, h, n = X_pred[1, :, :], X_pred[2, :, :], X_pred[3, :, :], X_pred[4, :, :]
    
    I_Na = G_NA .* (m.^3) .* h .* (V .- E_NA)
    I_K  = G_K  .* (n.^4) .* (V .- E_K)
    I_L  = G_L  .* (V .- E_L)
    
    f_V = (I_ext .- I_Na .- I_K .- I_L) ./ C_M
    f_m = alpha_m.(V) .* (1.0f0 .- m) .- beta_m.(V) .* m
    f_h = alpha_h.(V) .* (1.0f0 .- h) .- beta_h.(V) .* h
    f_n = alpha_n.(V) .* (1.0f0 .- n) .- beta_n.(V) .* n
    
    R_V, R_m, R_h, R_n = (dX_dt[1, :, :] .- f_V).^2, (dX_dt[2, :, :] .- f_m).^2, (dX_dt[3, :, :] .- f_h).^2, (dX_dt[4, :, :] .- f_n).^2
    L_steps = vec(mean(λ.RV .* R_V .+ λ.Rm .* R_m .+ λ.Rh .* R_h .+ λ.Rn .* R_n, dims=2))
    
    if use_causality && length(L_steps) > 1
        cum_losses = cumsum(L_steps)
        w = exp.(-ϵ .* cat(0.0f0, cum_losses[1:end-1]; dims=1))
        L_phys_total = sum(w .* L_steps)
    else
        L_phys_total = mean(L_steps)
    end
    
    L_ic, L_data = compute_boundary_and_data_losses(model, X_pred, batch_data)
    return L_phys_total + (λ.ic * L_ic) + (λ.data * L_data)
end


In [ ]:
# === Cell 7: Plot Predictions & Serialization Helper ===
function save_checkpoint_weights(save_path, model_id, seed, flat_params, metrics, loss_history)
    mkpath(dirname(save_path))
    checkpoint_data = Dict(
        "model_id" => model_id, "seed" => seed, "param_count" => length(flat_params),
        "weights" => Vector{Float64}(flat_params),
        "metrics" => Dict("l2_total" => Float64(metrics.l2_total), "mse_V" => Float64(metrics.mse_V)),
        "predictions" => Dict("V" => Vector{Float64}(metrics.V_pred), "m" => Vector{Float64}(metrics.m_pred), "h" => Vector{Float64}(metrics.h_pred), "n" => Vector{Float64}(metrics.n_pred)),
        "loss_history" => Vector{Float64}(loss_history)
    )
    open(save_path, "w") do io; JSON.print(io, checkpoint_data, 4); end
end

function evaluate_and_plot(model, HH_data, model_name)
    t_full_tensor = reshape(Float32.(HH_data.t), 1, 1, length(HH_data.t))
    full_prediction = model(t_full_tensor)
    
    V_pred, m_pred, h_pred, n_pred = full_prediction[1, 1, :], full_prediction[2, 1, :], full_prediction[3, 1, :], full_prediction[4, 1, :]
    V_gt, m_gt, h_gt, n_gt = Float32.(HH_data.V), Float32.(HH_data.m), Float32.(HH_data.h), Float32.(HH_data.n)
    
    l2_total = norm(V_pred .- V_gt) / norm(V_gt)
    mse_V = mean(abs2, V_pred .- V_gt)
    metrics = (l2_total=l2_total, mse_V=mse_V, V_pred=V_pred, m_pred=m_pred, h_pred=h_pred, n_pred=n_pred)

    default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=8, grid=true, frame=:box, lw=2.0)
    t = HH_data.t
    p_v = plot(t, HH_data.V, color=:black, line=:dash, label="Ground Truth")
    plot!(t, V_pred, color=:crimson, label="$model_name Prediction")
    title!("Membrane Potential Dynamics V(t)")
    ylabel!("Voltage (mV)")

    p_g = plot(t, HH_data.m, color=:black, line=:dash, label="GT m")
    plot!(t, m_pred, color=:dodgerblue, label="Model m")
    plot!(t, h_pred, color=:darkorange, label="Model h")
    plot!(t, n_pred, color=:forestgreen, label="Model n")
    title!("Ion Channel Gating Kinetics (m, h, n)")
    xlabel!("Time (ms)")
    ylabel!("Gating State")

    dashboard = plot(p_v, p_g, layout=grid(2, 1, heights=[0.5, 0.5]), link=:x, size=(800, 600))
    display(dashboard)
    return metrics
end


In [ ]:
# === Cell 6: Complete Dual-Stage Adam + L-BFGS Training Pipeline ===
Random.seed!(42)
println("=== Training Model M-MLP (Deterministic Seed=42) ===")

model = VanillaMLP(; hidden_dim=64, num_layers=4)

flat_params, _ = Flux.destructure(model)
println("Total Trainable Model Parameters: ", length(flat_params))

# Prepare Dataloader (Look-ahead horizon k=10, stride=5)
k = 10
stride = 5
max_idx = size(HH_data, 1) - (k - 1) * stride
t_train = reshape(Float32.(HH_data.t[1:max_idx]), 1, 1, max_idx)
gt_tensor = zeros(Float32, 4, k, max_idx)
for s in 1:max_idx
    for γ in 1:k
        r_idx = s + (γ - 1) * stride
        gt_tensor[1, γ, s] = Float32(HH_data.V[r_idx])
        gt_tensor[2, γ, s] = Float32(HH_data.m[r_idx])
        gt_tensor[3, γ, s] = Float32(HH_data.h[r_idx])
        gt_tensor[4, γ, s] = Float32(HH_data.n[r_idx])
    end
end
dataloader = Flux.DataLoader((t_train, gt_tensor), batchsize=64, shuffle=true)

# Stage 1: Adam Optimizer (1,000 Epochs)
println("\n=== STAGE 1: Adam Optimization (1000 Epochs) ===")
opt_adam = Flux.setup(Flux.Adam(1e-3), model)
λ_state = NTKState()
I_ext = 10.0f0
loss_history = Float32[]
best_loss = Inf32
best_weights = copy(flat_params)
milestones = Dict{Int, Vector{Float32}}()

for epoch in 1:1000
    epoch_loss = 0.0f0
    for (batch_t, batch_data) in dataloader
        if true
            update_ntk_weights_ema!(λ_state, model, batch_t, batch_data, I_ext; α=0.1f0)
        end
        l_val, grads = Flux.withgradient(m -> total_loss_causal(m, batch_t, batch_data, λ_state, I_ext), model)
        Flux.update!(opt_adam, model, grads[1])
        epoch_loss += l_val
    end
    push!(loss_history, epoch_loss)
    
    if epoch_loss < best_loss
        best_loss = epoch_loss
        w_curr, _ = Flux.destructure(model)
        best_weights = copy(w_curr)
    end
    
    if epoch in [100, 250, 500, 750, 1000]
        w_curr, _ = Flux.destructure(model)
        milestones[epoch] = copy(w_curr)
    end
    
    if epoch % 100 == 0 || epoch == 1
        println("Adam Epoch [$epoch / 1000] - Composite Causal Loss: ", round(epoch_loss, digits=4))
    end
end

# Stage 2: L-BFGS Refinement (200 Steps)
println("\n=== STAGE 2: L-BFGS Quasi-Newton Refinement (200 Iterations) ===")
p_flat, re = Flux.destructure(model)
s_t, s_data = first(dataloader)

function obj(p)
    l = total_loss_causal(re(p), s_t, s_data, λ_state, I_ext)
    push!(loss_history, l)
    return l
end
function grad!(g, p)
    _, grads = Flux.withgradient(m -> total_loss_causal(m, s_t, s_data, λ_state, I_ext), re(p))
    g_flat, _ = Flux.destructure(grads[1])
    g .= g_flat
end

res = Optim.optimize(obj, grad!, p_flat, LBFGS(m=10), Optim.Options(iterations=200, show_trace=false))
trained_model = re(Optim.minimizer(res))

println("\n=== TRAINING COMPLETE! ===")
metrics = evaluate_and_plot(trained_model, HH_data, "M-MLP")

# Save Checkpoints
checkpoint_dir = raw"C:\nirbhay\Downloads\NeuroPinnsFormmer-attention-that-neurons-needs\detailed_pinnsformmer_reports\checkpoints"
m_dir = joinpath(checkpoint_dir, "M-MLP", "seed_42")
save_checkpoint_weights(joinpath(m_dir, "checkpoint_final.json"), "M-MLP", 42, Optim.minimizer(res), metrics, loss_history)
save_checkpoint_weights(joinpath(m_dir, "checkpoint_best.json"), "M-MLP", 42, best_weights, metrics, loss_history)
